# 11 — Weather Enrichment `[Extension]`

**Airline Operations Intelligence Platform** · Notebook 11 · *runs locally*

## Purpose
Module 14 of the plan: join NOAA weather observations to flights, so the delay model can
see conditions it currently cannot.

This notebook closes two gaps at once:

1. **The model's biggest blind spot.** Notebook 06 reached ROC-AUC 0.663 using only
   airline, airports, schedule and historical rates. It cannot see fog, thunderstorms or
   snow — conditions that plainly cause delays.
2. **The `unstructured` data type (Unit 1).** The project has structured CSV and
   semi-structured BSON, but no unstructured data. NOAA's `REM` field carries raw
   **METAR text**, which §6 parses into features with regular expressions.

## Data source
NOAA Integrated Surface Database, hourly observations for 2015:
`https://www.ncei.noaa.gov/data/global-hourly/access/2015/`

Downloaded by `scripts/fetch_weather.py`, which is run **before** this notebook.

In [ ]:
import sys, re, subprocess
sys.path.insert(0, "../src")

from pathlib import Path
from config import build_spark, PATHS
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = build_spark("11-weather")

WEATHER_RAW = PATHS["root"] / "data" / "weather" / "raw"
STATIONS    = PATHS["root"] / "data" / "weather" / "stations.csv"

files = sorted(WEATHER_RAW.glob("*.csv"))
size_mb = sum(f.stat().st_size for f in files) / 1e6
print(f"Station files : {len(files)}  ({size_mb:.0f} MB)")

---
## 1. Station resolution

Airports and weather stations are different identifier spaces. The mapping is done in
`scripts/fetch_weather.py`:

- **Continental US:** ICAO = `K` + IATA (ATL → KATL), matched against NOAA's
  `isd-history.csv`. This resolved 57 of 60 airports.
- **Non-continental:** the pattern does not hold — Honolulu is **PHNL**, Kahului **PHOG**,
  San Juan **TJSJ**, none of which is `prefix + IATA`. Those fall back to the
  **nearest station by haversine distance**, which resolved all three within 1.2 km.

Assuming `K` + IATA universally would silently drop Hawaii and Puerto Rico — the same
class of error as the October airport codes in notebook 01.

In [ ]:
import pandas as pd
stations = pd.read_csv(STATIONS, dtype={"usaf": str, "wban": str})
print(f"Airports mapped : {len(stations)}")
print(stations["method"].str.split("_").str[0].value_counts().to_string())
print()
print(stations[stations.method != "icao"][["iata", "station", "method"]].to_string(index=False))

---
## 2. Loading — and a schema hazard worth knowing

ISD station files do **not** share a column set: sampled files ranged from **82 to 104
columns**, because optional observation groups (`AA1`–`AA4`, `AB1`, `GA1`…) appear only
when a station records them.

Spark's CSV reader takes its schema from the first file and aligns the rest **by
position**. Reading the directory in one call would therefore misalign every station whose
column set differs — silently producing wind speeds in the temperature column.

The fix is to read each file individually, select by **name**, and union.

In [ ]:
NEEDED = ["STATION", "DATE", "REPORT_TYPE", "WND", "CIG", "VIS", "TMP", "DEW", "SLP", "AA1", "REM"]

frames = []
for f in files:
    df = spark.read.option("header", True).csv(str(f))
    present = [c for c in NEEDED if c in df.columns]
    frames.append(df.select(*present))

obs = frames[0]
for df in frames[1:]:
    obs = obs.unionByName(df, allowMissingColumns=True)

obs = obs.cache()
print(f"Raw observations : {obs.count():,}")
obs.select("STATION", "DATE", "REPORT_TYPE", "WND", "TMP", "VIS").show(3, truncate=False)

In [ ]:
# FM-15 is the routine hourly METAR report. FM-16 (SPECI) are unscheduled special
# reports issued when conditions change abruptly -- keeping both would double-count hours.
obs.groupBy("REPORT_TYPE").count().orderBy(F.desc("count")).show(6)

hourly_obs = obs.filter(F.col("REPORT_TYPE") == "FM-15")
print(f"Hourly (FM-15) observations : {hourly_obs.count():,}")

---
## 3. Parsing composite fields

ISD packs several measurements into one comma-delimited string, and encodes *missing* as
a sentinel rather than a null:

| Field | Example | Meaning | Missing sentinel |
|---|---|---|---|
| `WND` | `330,1,N,0031,1` | direction°, quality, type, speed m/s ×10 | dir `999`, speed `9999` |
| `TMP` | `+0061,1` | 6.1 °C | `+9999` |
| `DEW` | `+0006,1` | 0.6 °C dewpoint | `+9999` |
| `VIS` | `016000,1,9,9` | 16,000 m visibility | `999999` |
| `CIG` | `22000,5,9,N` | cloud ceiling m | `99999` |
| `AA1` | `01,0000,9,5` | period h, precipitation depth mm ×10 | depth `9999` |

**Treating the sentinels as numbers would be catastrophic**: a missing temperature would
enter the model as +999.9 °C and a missing visibility as 999 km. Every parse below
converts its sentinel to null explicitly.

In [ ]:
def part(col, i):
    return F.split(F.col(col), ",").getItem(i)

def scaled(col, i, sentinel, divisor=10.0):
    """Parse one component, mapping the ISD sentinel to NULL rather than a number."""
    v = part(col, i).cast("double")
    return F.when(v == F.lit(sentinel), None).otherwise(v / divisor)

weather = (hourly_obs
    .withColumn("wind_dir",    F.when(part("WND", 0).cast("double") == 999, None)
                                .otherwise(part("WND", 0).cast("double")))
    .withColumn("wind_speed",  scaled("WND", 3, 9999))       # m/s
    .withColumn("temp_c",      scaled("TMP", 0, 9999))
    .withColumn("dewpoint_c",  scaled("DEW", 0, 9999))
    .withColumn("visibility_m", F.when(part("VIS", 0).cast("double") == 999999, None)
                                 .otherwise(part("VIS", 0).cast("double")))
    .withColumn("ceiling_m",   F.when(part("CIG", 0).cast("double") == 99999, None)
                                .otherwise(part("CIG", 0).cast("double")))
    .withColumn("precip_mm",   F.when(F.col("AA1").isNull(), F.lit(0.0))
                                .otherwise(scaled("AA1", 1, 9999)))
    .withColumn("obs_ts",      F.to_timestamp("DATE", "yyyy-MM-dd'T'HH:mm:ss")))

weather.select("temp_c", "dewpoint_c", "wind_speed", "visibility_m",
               "ceiling_m", "precip_mm").describe().show()

In [ ]:
# Assertion: no sentinel may survive into a feature column.
bad = weather.filter(
    (F.col("temp_c") > 100) | (F.col("temp_c") < -90) |
    (F.col("wind_speed") > 120) | (F.col("visibility_m") > 200000) |
    (F.col("precip_mm") > 500)
).count()
print(f"Rows with impossible values after parsing : {bad:,}")
assert bad == 0, "an ISD sentinel leaked into a feature column"
print("PASS -- sentinels handled; every value is physically plausible.")

---
## 4. Unstructured data: parsing METAR text

The `REM` field holds the **raw METAR report** — free text written for human pilots, not
for machines:

```
METAR KATL 010052Z 33005KT 10SM FEW200 SCT250 05/01 A3037 RMK AO2 SLP289 T00500006
```

This is the project's **unstructured** data source (Unit 1, Types of data). Turning it into
model features means regular-expression extraction of the standard weather phenomena codes,
which appear in no numeric column.

In [ ]:
metar = weather.filter(F.col("REM").isNotNull() & F.col("REM").contains("METAR"))
print(f"Observations carrying METAR text : {metar.count():,}\n")
for r in metar.select("REM").limit(3).collect():
    print("  ", r["REM"][:150])

In [ ]:
# METAR phenomena codes, extracted from free text into boolean features.
PHENOMENA = {
    "wx_thunderstorm": r"\bTS(RA|SN|GR)?\b",
    "wx_snow":         r"(?<![A-Z])(\+|-)?SN\b",
    "wx_rain":         r"(?<![A-Z])(\+|-)?RA\b",
    "wx_fog":          r"\b(FG|BR)\b",
    "wx_freezing":     r"\bFZ(RA|DZ|FG)\b",
    "wx_haze_smoke":   r"\b(HZ|FU)\b",
}

weather_wx = weather
for name, pattern in PHENOMENA.items():
    weather_wx = weather_wx.withColumn(
        name, F.when(F.col("REM").isNull(), 0)
               .otherwise((F.regexp_extract(F.col("REM"), pattern, 0) != "").cast("int")))

counts = weather_wx.agg(*[F.sum(c).alias(c) for c in PHENOMENA]).first().asDict()
total = weather_wx.count()
print(f"{'PHENOMENON':<20}{'OBSERVATIONS':>14}{'% OF HOURS':>12}")
print("-" * 46)
for k, v in sorted(counts.items(), key=lambda kv: -(kv[1] or 0)):
    print(f"{k:<20}{v or 0:>14,}{100*(v or 0)/total:>11.2f}%")
print("\nThese features exist nowhere in the numeric columns -- they were recovered")
print("from free text, which is what makes this genuinely unstructured data.")

---
## 5. Aggregate to airport-hour

Multiple reports can land in the same hour. Aggregating to one row per
(airport, date, hour) gives a clean join key and averages away sensor noise.

In [ ]:
station_map = spark.createDataFrame(
    stations.assign(station_id=stations.usaf + stations.wban)[["station_id", "iata"]])

hourly = (weather_wx
    .withColumn("station_id", F.col("STATION"))
    .join(F.broadcast(station_map), "station_id", "inner")
    .withColumn("obs_date", F.to_date("obs_ts"))
    .withColumn("obs_hour", F.hour("obs_ts"))
    .groupBy(F.col("iata").alias("wx_airport"), "obs_date", "obs_hour")
    .agg(F.avg("temp_c").alias("temp_c"),
         F.avg("dewpoint_c").alias("dewpoint_c"),
         F.avg("wind_speed").alias("wind_speed"),
         F.min("visibility_m").alias("visibility_m"),      # worst visibility in the hour
         F.min("ceiling_m").alias("ceiling_m"),            # lowest ceiling in the hour
         F.max("precip_mm").alias("precip_mm"),
         *[F.max(c).alias(c) for c in PHENOMENA]))

hourly = hourly.cache()
print(f"Airport-hour observations : {hourly.count():,}")
print(f"Airports covered          : {hourly.select('wx_airport').distinct().count()}")
hourly.orderBy("obs_date", "obs_hour").show(4)

---
## 6. Join to flights

LEFT join on origin airport, flight date and scheduled departure hour, with the unmatched
count reported explicitly — the same discipline as notebook 02. Weather is joined for the
**origin only**: destination weather hours ahead is not known at departure time, and using
it would be a subtle form of leakage.

In [ ]:
flights = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))
N = flights.count()

enriched = (flights.join(
    hourly,
    (flights.origin == hourly.wx_airport) &
    (flights.flight_date == hourly.obs_date) &
    (flights.sched_dep_hour == hourly.obs_hour),
    "left").drop("wx_airport", "obs_date", "obs_hour"))

matched = enriched.filter(F.col("temp_c").isNotNull()).count()
print(f"Flights total          : {N:,}")
print(f"With weather matched   : {matched:,}  ({100*matched/N:.1f}%)")
print(f"Without                : {N-matched:,}  ({100*(N-matched)/N:.1f}%)")
assert enriched.count() == N, "the join changed the row count"
print("\nRow count preserved. Unmatched flights keep null weather and fall back to the")
print("global mean in notebook 06, exactly as unseen routes already do.")

In [ ]:
# Coverage is bounded by the 60 airports downloaded, by design.
covered = hourly.select("wx_airport").distinct()
by_airport = (enriched.groupBy("origin")
    .agg(F.count("*").alias("flights"),
         F.round(100.0 * F.avg(F.col("temp_c").isNotNull().cast("int")), 1).alias("wx_coverage_pct"))
    .orderBy(F.desc("flights")))
by_airport.show(10)
print("Airports outside the downloaded set show 0% coverage by design -- the download")
print("targets the busiest 60, which carry the large majority of flights.")

---
## 7. Does weather actually relate to delay?

Before adding these features to a model, check that they carry signal at all. A feature
with no relationship to the target adds variance and nothing else.

In [ ]:
w = enriched.filter(F.col("temp_c").isNotNull() & (F.col("status") == "completed"))

print("Delay rate by weather phenomenon (origin, at scheduled departure hour):\n")
print(f"{'CONDITION':<22}{'FLIGHTS':>12}{'DELAY RATE':>12}{'vs BASE':>10}")
print("-" * 56)
base = w.agg(F.avg("is_delayed")).first()[0]
print(f"{'(all matched flights)':<22}{w.count():>12,}{100*base:>11.2f}%{'':>10}")
for name in PHENOMENA:
    sub = w.filter(F.col(name) == 1)
    n = sub.count()
    if n < 500:
        continue
    rate = sub.agg(F.avg("is_delayed")).first()[0]
    print(f"{name:<22}{n:>12,}{100*rate:>11.2f}%{100*(rate-base):>+9.2f}pp")

In [ ]:
# Continuous features, bucketed.
print("\nDelay rate by visibility:")
(w.withColumn("vis_band",
    F.when(F.col("visibility_m") < 1600, "1. under 1.6 km")
     .when(F.col("visibility_m") < 4800, "2. 1.6-4.8 km")
     .when(F.col("visibility_m") < 9999, "3. 4.8-10 km")
     .otherwise("4. 10 km or more"))
 .groupBy("vis_band")
 .agg(F.count("*").alias("flights"),
      F.round(100*F.avg("is_delayed"), 2).alias("delay_rate_pct"))
 .orderBy("vis_band").show(truncate=False))

print("Delay rate by wind speed:")
(w.withColumn("wind_band",
    F.when(F.col("wind_speed") < 3, "1. calm (<3 m/s)")
     .when(F.col("wind_speed") < 8, "2. moderate (3-8)")
     .when(F.col("wind_speed") < 13, "3. strong (8-13)")
     .otherwise("4. gale (13+)"))
 .groupBy("wind_band")
 .agg(F.count("*").alias("flights"),
      F.round(100*F.avg("is_delayed"), 2).alias("delay_rate_pct"))
 .orderBy("wind_band").show(truncate=False))

---
## 8. Write outputs

In [ ]:
out = PATHS["curated"] / "flights_weather.parquet"
(enriched.write.mode("overwrite").partitionBy("month").parquet(str(out)))
print("Wrote", out)

size = subprocess.run(["du", "-sh", str(out)], capture_output=True, text=True).stdout.split()[0]
print("Size on disk:", size)

check = spark.read.parquet(str(out))
assert check.count() == N, "row count changed on write"
print(f"Verified: {check.count():,} rows, {len(check.columns)} columns "
      f"({len(check.columns) - len(flights.columns)} weather columns added)")

In [ ]:
# Keep a sample of raw METAR text as evidence of the unstructured source.
(metar.select(F.col("STATION").alias("station_id"), F.col("DATE").alias("observed_at"),
              F.col("REM").alias("metar_text"))
      .limit(500).coalesce(1)
      .write.mode("overwrite").parquet(str(PATHS["marts"] / "weather_text_samples.parquet")))
print("Wrote weather_text_samples.parquet (raw METAR text, for the Variety section)")

---
## 9. Summary

| Output | Purpose |
|---|---|
| `data/curated/flights_weather.parquet` | Flights + weather, input to notebook 06 |
| `data/marts/weather_text_samples.parquet` | Raw METAR text, evidence of unstructured data |

### What this adds to the syllabus

- **Unit 1 — Variety:** the project now handles structured (CSV), semi-structured (BSON)
  **and unstructured** (METAR free text) data, rather than documenting the third as absent.
- **Unit 1 — Veracity:** ISD sentinel encoding is a textbook trap; §3 asserts none survives.
- **Unit 5 — Preprocessing:** composite-field parsing, multi-source joins on a compound
  key, and regex feature extraction from text.

### Honest limitations

- **Coverage is bounded by the 60 downloaded airports.** Flights from smaller airports have
  no weather and fall back to the global mean.
- **Origin weather only.** Destination weather at arrival time is not known at departure,
  so including it would be leakage.
- **Hourly resolution.** A thunderstorm that clears in 20 minutes still marks the hour.
- **Correlation, not causation.** Bad weather coincides with delay, but a low-visibility
  hour at a congested hub confounds weather with congestion.

### Next
`06_ml_classification.ipynb` — retrain with these features and measure whether they
actually help, in the ablation table.

In [ ]:
for df in (obs, hourly):
    df.unpersist()
spark.stop()
print("Notebook 11 complete.")